# Orislop AV Joint v1
Use a T4 for the smoke test only. Use an A100 for real 100-speaker/50-hour training. The release trainer validates commercial rights and starts from scratch.

In [ ]:
from pathlib import Path
import os, subprocess, zipfile
from google.colab import drive
REPO = Path('/content/Orislop-landing')
UPLOAD_BUNDLE = Path('/content/orislop-av-joint-colab-upload.zip')
if UPLOAD_BUNDLE.is_file():
    REPO.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(UPLOAD_BUNDLE) as archive:
        archive.extractall(REPO)
    print('Loaded the manually uploaded AV Joint training bundle.')
elif not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/coolguy860/Orislop-landing.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', 'training/orislop_av_joint/requirements-colab.txt'], check=True)
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
drive.mount('/content/drive')
print('Repository ready at', REPO)

In [ ]:
!python training/orislop_av_joint/train.py self-test

Add the corpus, manifest, rights ledger, and YuNet model to the Drive paths below. Review the rights report before continuing. Legitimate dubbing/delay must be sync mismatch = 1 and joint forgery = 0.

In [ ]:
MANIFEST = '/content/drive/MyDrive/orislop-av-joint/manifest.jsonl'
RIGHTS = '/content/drive/MyDrive/orislop-av-joint/rights.json'
DATA = '/content/drive/MyDrive/orislop-av-joint/data'
PREPARED = '/content/drive/MyDrive/orislop-av-joint/prepared'
YUNET = '/content/drive/MyDrive/orislop-av-joint/models/face_detection_yunet.onnx'
RUN = '/content/drive/MyDrive/orislop-av-joint/runs/av_joint_v1.pt'

In [ ]:
!python training/orislop_av_joint/train.py validate-rights --manifest "{MANIFEST}" --rights-ledger "{RIGHTS}" --output /content/rights_report.json
!python training/orislop_av_joint/prepare.py --manifest "{MANIFEST}" --data-root "{DATA}" --output-root "{PREPARED}" --yunet-model "{YUNET}" --seconds 8

In [ ]:
PREPARED_MANIFEST = f'{PREPARED}/prepared_manifest.jsonl'
!python training/orislop_av_joint/train.py train --manifest "{PREPARED_MANIFEST}" --rights-ledger "{RIGHTS}" --config configs/av_joint_v1.json --output "{RUN}" --epochs 20 --batch-size 4

In [ ]:
!python training/orislop_av_joint/train.py evaluate --manifest "{PREPARED_MANIFEST}" --checkpoint "{RUN}" --split test --output /content/av_test_metrics.json
!python training/orislop_av_joint/train.py calibrate --manifest "{PREPARED_MANIFEST}" --checkpoint "{RUN}" --output /content/temperatures.json
!python training/orislop_av_joint/train.py export --checkpoint "{RUN}" --temperatures /content/temperatures.json --output-dir /content/av_joint_artifact